In [ ]:
from langchain_community.utilities import SQLDatabase
from dataclasses import dataclass
from langchain_core.tools import tool
from langgraph.runtime import get_runtime
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
db= SQLDatabase.from_uri("sqlite:///Chinook.db")

In [ ]:
@dataclass
class RunTimeContext:
    db: SQLDatabase

In [ ]:
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results"""
    runtime= get_runtime(RunTimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"
    


In [ ]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [ ]:
agent= create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    system_prompt= SYSTEM_PROMPT,
    context_schema= RunTimeContext
)

In [ ]:
question= "This is Frank Harris, What was the total on my last invoice?"
steps=[]

for step in agent.stream(
    {"messages": [{"role": "user", "context": question}]},
    stream_mode= "values",
    context= RunTimeContext(db= db)
):  
    step["messages"][-1].pretty_print
    steps.append(step)

Adicionando memória curta

In [ ]:
agent= create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    system_prompt= SYSTEM_PROMPT,
    context_schema= RunTimeContext(db = db),
    checkpointer= InMemorySaver()
)

In [ ]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context= RunTimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

In [ ]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context= RunTimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)